# ⚡ 12 · 专家卸载（Expert Offloading）与 CPU-GPU 协同

> AI-Infra 面试高频：**显存装不下整个 MoE 模型时，怎么把「上一秒算过的专家」换成「下一秒要算的专家」，还不让 GPU 干等？**

- 稀疏 MoE 的参数量全在专家里：Mixtral 8x7B 总参数 46.7B，推理时每 token 只激活 `top-2`，**算力只用一个零头，显存却要装下全部**；
- 装不下就卸载（offload）到 CPU DRAM / NVMe，按路由按需搬运 —— 但 PCIe 只有几十 GB/s，于是瓶颈从 GPU 算力变成**搬运带宽**；
- 工程答案是一整套协同：**卸载比例账本 + 双缓冲 + 层级预取 + 热点专家常驻 + 量化压缩**。本节把这条链路算清楚、模拟出来。

## 📋 本节要解决的问题

| # | 问题 | 答案 |
|---|---|---|
| 1 | 为什么要卸载专家？ | 参数全在专家里，激活只有 top-k；显存装不下全量专家权重 |
| 2 | 卸载多少合适？ | 账本 $r=1-(B-P_{\text{dense}})/P_{\text{exp}}$，剩下靠搬运补 |
| 3 | 卸载后瓶颈在哪？ | PCIe 带宽；上界 $\text{tokens/s}\le BW/(rH)$；再与算力上限取小 |
| 4 | 怎么不让 GPU 干等？ | 双缓冲 + 双层流水：算第 $i$ 层时搬第 $i+1$ 层 |
| 5 | 搬运和计算怎么并行？ | 两个硬件单元 + 两个队列，深度 2 的环形缓冲槽 |
| 6 | 三种调度差多少？ | 逐层同步 → 静态预取 → 动态路由触发，重叠率逐级变化 |
| 7 | 批量能不能救？ | 能：batch 个 token 共享一次搬运，把瓶颈从带宽推回算力 |
| 8 | 业界怎么做的？ | Fiddler / DeepSpeed-MoE / KTransformers / QMoE / ZeRO-Offload |

## 1️⃣ 直觉与大白话

### 类比一：书架只放当天要读的书

你有一个书桌（GPU 显存，24 GB）和一个大书柜（CPU DRAM / SSD，几百 GB）。

- **全量专家权重** = 整个书柜的书（Mixtral 的 45.1 GB 专家）；
- 书桌放不下全部，于是**只把今天要读的几摞搬到桌上**（常驻专家），其余留在书柜（卸载专家）；
- 每读一章（一层 Layer）需要的书不一样，于是读完第 $i$ 章、去书柜取第 $i+1$ 章的书 —— **取书这段时间只能干等**，这就是 `stall`；
- 聪明的做法：**第 $i$ 章还在读时，就让人先把第 $i+1$ 章的书搬过来**（双缓冲 + 预取），取书时间被阅读时间掩盖 → 这就是 `overlap`；
- 桌上**只有两个空位**轮流用（深度 2 双缓冲）：一个正在读，一个正在搬。

一句话：卸载的本质是**「内存不够，用时间换空间」**；协同的本质是**「尽量让这段多花的时间被别人吃掉」**。

### 类比二：后厨备菜与炒菜重叠

- 灶台 = GPU 计算单元（SM），切配台 = PCIe 搬运通路（DMA）；
- 一个厨师体系里，**切菜的人和炒菜的人可以同时干活**；
- 但如果切菜的太慢（PCIe 只有 8 GB/s），炒菜的每炒完一道就得等下一盘配菜 → 锅空转（stall）；
- 解决办法：① 多备一些常驻的「常用菜」（热点专家常驻）；② 按菜单提前切下一道（layer-wise 预取）；③ 菜切得更小份（量化，2-bit 让体积降到 1/8）。

三条结论，本节后面都会用数字兑现：

1. **卸载比例由显存预算唯一确定**，不是拍脑袋；
2. **只要搬运时间 > 计算时间，重叠就只能掩盖一部分**，剩余部分变成 stall —— 所以卸载后吞吐的天花板是 PCIe 带宽；
3. **批量（batch）是免费的杠杆**：$B$ 个 token 共享一次专家搬运，计算量涨 $B$ 倍、搬运量不变 → 把瓶颈从带宽推回算力。

### 为什么 MoE 天生要卸载：参数、激活、显存三者对不上

| 维度 | Mixtral 8x7B（专家 FP8 · dense BF16） | 谁在为它买单 |
|---|---|---|
| 总参数 | 46.7 B（BF16 ≈93 GB；专家量化到 FP8 后 ≈47 GB） | **显存**：必须装下（或用卸载换空间） |
| 每 token 激活参数 | ≈12.9 B | **算力**：只按这个花 FLOPs |
| 专家权重（FP8） | 45.1 GB（同理 45.1 G 参数；BF16 口径 ≈90 GB，见 11 篇） | 显存里最重的那一块，也是卸载的主战场 |
| 每 token 专家命中 | 32 层 × top-2 = 64 次 | **搬运量**：决定 PCIe 压力 |
| 稠密对照（13B） | 26 GB、每 token 花 13B 计算 | 装得下就不需要本节的技术 |

> **口径说明（很重要）**：本节统一按「**专家权重 FP8（1 B/参数）、dense 权重 BF16（2 B/参数）**」计算——这等价于把「量化卸载（QMoE 思路）」一起算进来，也是 24 GB 单卡能碰 MoE 的现实前提。若专家按纯 BF16 存放，账本、每 token 搬运字节与带宽上界全部 **×2**（专家总量 ≈90 GB、单专家 ≈352 MB），与 [11-专家分页与显存池化](11-专家分页与显存池化.ipynb) 的 BF16 账本互为对照；结论的形状（r、重叠率、tokens/s 随带宽单调）完全不变。

一句话：**稀疏性兑现的是算力红利，不是显存红利**。MoE 用「12.9B 的计算」买到「46.7B 的表达力」，代价是显存必须按 46.7B 准备 —— 装不下时，就只能把这笔账挪到带宽上。

这也解释了为什么卸载几乎只出现在 MoE 场景：稠密模型的参数量与激活量同阶，卸载它等于「每个 token 都把整个模型搬一遍」，毫无意义。

In [ ]:
# ---- matplotlib 中文字体回退（防图内中文乱码）----
import matplotlib

matplotlib.rcParams["font.sans-serif"] = [
    "Microsoft YaHei",
    "SimHei",
    "Songti SC",
    "Arial Unicode MS",
    "DejaVu Sans",
]
matplotlib.rcParams["axes.unicode_minus"] = False  # 负号用 ASCII，避免方框
# ---------------------------------------------------
import os

import matplotlib.pyplot as plt
import numpy as np

SEED = 42
np.random.seed(SEED)
plt.close("all")

# ===== 模型：Mixtral 8x7B（专家 FP8、dense BF16；量化卸载口径）=====
L_LAYERS = 32          # 层数
N_EXPERTS = 8          # 每层专家数
TOP_K = 2              # 每 token 激活专家数
P_EXP_GB = 45.1        # 全部专家权重（FP8，1 B/参数；BF16 则 ≈90.2 GB）
P_DENSE_GB = 1.9       # attention / embedding / router / norm 常驻
EXPERT_GB = P_EXP_GB / (L_LAYERS * N_EXPERTS)   # 单个专家权重

# ===== 硬件参数（全部可改）=====
HBM_BW_GB_S = 900.0    # GPU HBM 带宽
ACTIVE_GB = 25.8       # decode 每 token 需读取的激活权重（≈12.9B×2B）
BW_LIST = [8, 16, 32, 64]        # PCIe 有效带宽 GB/s
RES_LIST = [0.3, 0.5, 0.7]       # 专家权重驻留比例 f = 1 - r

HANDLED_GB = L_LAYERS * TOP_K * EXPERT_GB   # 每 token 命中的专家字节（裸搬运量）

print(f"[模型] 单专家权重       : {EXPERT_GB * 1024:7.1f} MB   （共 {L_LAYERS * N_EXPERTS} 个）")
print(f"[模型] 每 token 路由命中 : {L_LAYERS * TOP_K:3d} 个专家 → 裸搬运 {HANDLED_GB:.2f} GB")
print(f"[硬件] decode 访存/token : {ACTIVE_GB:.1f} GB → 算力侧天花板 {(HBM_BW_GB_S / ACTIVE_GB):.1f} tokens/s")
print(f"[账本] 全驻留需求        : {P_DENSE_GB + P_EXP_GB:.1f} GB "
      f"（dense {P_DENSE_GB} + experts {P_EXP_GB}）")

## 2️⃣ 卸载比例与显存账本

### 推导：显存预算决定卸载比例

记显存预算 $B$、常驻非专家参数 $P_{\text{dense}}$、专家参数总量 $P_{\text{exp}}$，专家驻留比例 $f$、卸载比例 $r = 1-f$。

显存里必须装下「常驻部分 + 留在 GPU 上的那部分专家」：

$$ P_{\text{dense}} + f\cdot P_{\text{exp}} \le B \quad\Longrightarrow\quad f \le \frac{B - P_{\text{dense}}}{P_{\text{exp}}} $$

因此**恰好用满预算**时的卸载比例：

$$ \boxed{\,r = 1 - \frac{B - P_{\text{dense}}}{P_{\text{exp}}}\,} $$

三个边界要会解释：

| 条件 | 结论 | 含义 |
|---|---|---|
| $B \ge P_{\text{dense}} + P_{\text{exp}}$ | $r = 0$ | 装得下，无需卸载 |
| $B = P_{\text{dense}}$ | $r = 1$ | 连一个专家都放不下，纯流式 |
| $B < P_{\text{dense}}$ | 不可行 | 连稠密部分都放不下，只能换卡或量化 |

### 记号表

| 符号 | 含义 | 本节取值 |
|---|---|---|
| $B$ | 显存预算（可用给权重） | 16 / 24 / 40 / 80 GB |
| $P_{\text{dense}}$ | 常驻非专家参数 | 1.9 GB |
| $P_{\text{exp}}$ | 专家参数总量 | 45.1 GB |
| $r$ | 卸载比例 | 由上式算出 |
| $f = 1-r$ | 驻留比例 | 0.3 / 0.5 / 0.7（实验扫描） |
| $H$ | 每 token 命中的专家字节 | 11.27 GB |

**关键直觉**：卸载比例的分子分母都是**字节**，所以「换更好显卡」和「把权重压到 2-bit」在这个公式里是同一件事 —— 后者等于把 $P_{\text{exp}}$ 缩小 8 倍。

In [ ]:
def unload_ratio(budget_gb, p_dense=P_DENSE_GB, p_exp=P_EXP_GB):
    '''r = 1 - (B - P_dense) / P_exp，裁剪到 [0, 1]。

    返回 (r, 常驻专家字节 GB)。
    '''
    resident = float(np.clip(budget_gb - p_dense, 0.0, p_exp))
    r = 1.0 - resident / p_exp
    return float(np.clip(r, 0.0, 1.0)), resident


def ledger(budget_gb):
    '''返回 (r, 常驻专家 GB, 权重总占用 GB)，并校验账本闭合。'''
    r, resident = unload_ratio(budget_gb)
    total = P_DENSE_GB + resident
    expect = min(budget_gb, P_DENSE_GB + P_EXP_GB)
    assert abs(total - expect) < 1e-9, (budget_gb, total, expect)   # 账本必须闭合
    assert 0.0 <= r <= 1.0
    return r, resident, total


def per_token_transfer_gb(r):
    '''每 token 必须从片外搬进显存的专家字节：只有「被卸载」的那部分要动。

    驻留比例 f = 1 - r，故 搬运量 = (1 - f) · H = r · H。
    '''
    return r * HANDLED_GB


# 三个边界的自检：装得下 → r=0；刚好放 dense → r=1
r_zero, _, _ = ledger(P_DENSE_GB + P_EXP_GB)
r_one, _, _ = ledger(P_DENSE_GB)
print(f"装得下（B = dense+exp = {P_DENSE_GB + P_EXP_GB:.1f} GB） → r = {r_zero:.4f}")
print(f"刚放 dense（B = {P_DENSE_GB} GB）                → r = {r_one:.4f}")
assert abs(r_zero) < 1e-9 and abs(r_one - 1.0) < 1e-9
print("边界自检 PASS")

In [ ]:
print(f"{'预算B':>7} {'可给专家':>9} {'卸载比例r':>10} {'常驻专家':>9} {'每token搬运':>12} {'@32GB/s上界':>13} {'余量':>8}")
rows_budget = []
for B in (16, 24, 40, 80):
    r, resident, total = ledger(B)
    d = per_token_transfer_gb(r)
    cap32 = 32.0 / d if d > 0 else float("inf")
    spare = max(0.0, B - total)
    rows_budget.append((B, r, resident, d, cap32))
    cap_str = f"{cap32:.2f}" if np.isfinite(cap32) else "无需搬运"
    print(f"{B:>5}GB {B - P_DENSE_GB:>8.1f} {r * 100:>9.2f}% {resident:>8.1f} "
          f"{d:>11.2f}GB {cap_str:>13} {spare:>7.1f}GB")

# ---------- 主线案例：24 GB 卡跑 Mixtral 8x7B ----------
r24, res24, _ = ledger(24.0)
d24 = per_token_transfer_gb(r24)
print("\n【24 GB 卡跑 Mixtral 8x7B（专家 FP8 · dense BF16）】")
print(f"  卸载比例 r = 1 - (24 - 1.9) / 45.1 = {r24:.4f}  → 卸载 {r24 * 100:.2f}%、驻留 {100 - r24 * 100:.2f}%")
print(f"  常驻专家 {res24:.2f} GB（= 24 - 1.9，预算恰好用满）")
print(f"  每 token 搬运 {d24:.2f} GB（裸搬运 {HANDLED_GB:.2f} GB，只需搬 {r24 * 100:.2f}%）")
print(f"  即：约 {r24 * 100:.0f}% 的专家权重留在 CPU 侧，每生成 1 个 token 要搬 {d24:.2f} GB")

assert 0.50 <= r24 <= 0.52          # 24GB 卡卸载比例约 51%
assert res24 <= 24.0 - P_DENSE_GB + 1e-9

## 2️⃣（续）带宽受限上界：卸载后吞吐的天花板

GPU 每算一层，就要把这层命中但没常驻的专家搬进来。搬运和计算可以并行，所以每 token 的串行时间下界是两者取大：

$$ \text{每 token 时间} \;\ge\; \max\Big(\underbrace{\frac{r\,H}{BW}}_{\text{搬运}},\;\underbrace{\frac{A}{\text{HBM}_{BW}}}_{\text{算力}}\Big) $$

其中 $H$ 是「每 token 命中的专家字节」（Mixtral 8x7B 为 11.28 GB），$rH$ 是**真正要动的字节**——只有被卸载的那部分才需要搬。于是带宽受限上界：

$$ \boxed{\;\text{tokens/s} \;\le\; \frac{BW}{\text{每 token 搬运字节}} \;=\; \frac{BW}{r\,H} \;=\; \frac{BW}{(1-f)\,H}\;} $$

其中 $f = 1-r$ 记作**有效驻留比例**。面试口语常把它说成「带宽 × 有效驻留 ÷ 每 token 搬运字节」——注意分子上的驻留比例与分母 $H$ 的口径必须一致，本节统一写成 $BW/(rH)$，**不把驻留比例重复计入**：否则 $r\to 0$（全部驻留）时上界会错成一个有限值，而物理上此时根本没有搬运需求。

几个必须能脱口而出的推论：

1. **$r$ 与 $BW$ 可互换**：卸载比例减半（$r:0.5\to0.25$）与带宽翻倍在上界上是同一件事 —— 这也是为什么「多买一张卡做 EP/TP」和「少卸载一点」在吞吐上常常等价；
2. **batch 摊薄搬运**：$B$ 个 token 共享一次专家搬运，上界变成 $B\cdot BW/(rH)$，直到撞上算力天花板 $\text{HBM}_{BW}/A$；
3. **量化直接改分子**：2-bit 让 $rH$ 降到约 $1/8$，上界抬 8 倍 —— 这就是 QMoE 的思路；
4. **该式是上界不是预测**：它假设搬运 100% 被掩盖（零 stall），真实值由第 5 节的流水模拟给出。

### 四个极端预算的直觉校验

把公式推到边界，比记住公式更能检验理解：

| 预算 $B$ | $r$ | 物理含义 | 面试时的说法 |
|---|---|---|---|
| 80 GB | 0% | 装得下全部 → 一次都不用搬 | 此时该做的是**提高 batch**，而不是省显存 |
| 47 GB | 0%（临界） | 恰好装下，零余量 | 任何 KV cache 增长都会立刻逼出卸载 |
| 24 GB | 51% | 约一半专家在对岸 | 每 token 搬 5.75 GB，吞吐被 PCIe 钉住 |
| 1.9 GB | 100% | 连一个专家都放不下 → 纯流式 | 等价于把专家当「外部数据库」逐 token 查询 |

注意 47 GB 那一行的脆弱性：账本算的是**稳态**，而真实推理里 KV cache 随上下文增长 —— 这就是为什么生产配置总要预留余量，而不是把 $B$ 顶到极限。

In [ ]:
def tps_bound(bw_gb_s, r, batch=1):
    '''两条上界取小，返回 (上界, 纯带宽上界)。

    ① 带宽受限：tokens/s ≤ BW / (每 token 搬运字节) = BW / (r·H)
    ② 算力受限：tokens/s ≤ HBM_BW / A（与 batch 无关）
    batch 个 token 共享一次专家搬运，故 ① 随 batch 线性放宽。
    '''
    d = per_token_transfer_gb(r)
    bw_cap = batch * bw_gb_s / d if d > 0 else float("inf")
    cpu_cap = HBM_BW_GB_S / ACTIVE_GB
    return min(bw_cap, cpu_cap), bw_cap


cpu_ceiling = HBM_BW_GB_S / ACTIVE_GB
print(f"算力侧天花板 = HBM {HBM_BW_GB_S:.0f} GB/s ÷ {ACTIVE_GB:.1f} GB/token = {cpu_ceiling:.1f} tokens/s")
print(f"\n带宽受限上界（batch=1，单位 tokens/s）：")
print(f"{'BW(GB/s)':>9} " + " ".join(f"{'f=' + str(f):>10}" for f in RES_LIST))
for bw in BW_LIST:
    line = " ".join(f"{tps_bound(bw, 1.0 - f)[0]:>10.2f}" for f in RES_LIST)
    print(f"{bw:>9} {line}")

print("\n【24 GB 卡（r=0.51，f=0.49）】")
for bw in BW_LIST:
    cap, bw_cap = tps_bound(bw, r24)
    print(f"  BW={bw:>2} GB/s → 带宽上界 {bw_cap:>6.2f} tokens/s，取 min(带宽, 算力) = {cap:.2f} tokens/s")
print("\n⚠ batch=1 时上界只有 1~12 tokens/s：这是「搬运瓶颈」，不是 GPU 慢。")
assert tps_bound(64, r24)[0] > tps_bound(8, r24)[0]      # 上界随 BW 递增
assert tps_bound(8, r24)[0] <= cpu_ceiling + 1e-9

## 2️⃣（续）预算不是 24 GB：真实可用显存要一层层扣

账本里的 $B$ 必须是**真正能留给权重的**那部分。24 GB 的卡上，先被吃掉的是：

| 项目 | 典型占用 | 说明 |
|---|---|---|
| CUDA context / driver / 显存碎片 | ~1.5 GB | 进程级固定开销 |
| KV cache（batch 8 × 1k ctx） | ~1.05 GB | 随上下文线性增长，长上下文是主要杀手 |
| 激活值 / workspace | ~1.0 GB | 中间张量、cuBLAS workspace |

所以真实权重预算约 $20.4$ GB，卸载比例会被进一步推高。**这一格经常是面试的加分项**：只说「24 GB 装 45 GB 所以要卸载 53%」是不够的，要能指出 $B$ 本身需要打折。

In [ ]:
budget_total = 24.0
components = [
    ("CUDA context / driver / 碎片", 1.5),
    ("KV cache（batch 8, 1k ctx）", 1.05),
    ("激活值 / workspace", 1.0),
]
used = sum(v for _, v in components)
avail = budget_total - used
r_real, resident_real, _ = ledger(avail)
d_real = per_token_transfer_gb(r_real)

print(f"权重预算 B = {budget_total:.1f} GB")
for name, v in components:
    print(f"   - {v:>5.2f} GB  {name}")
print(f"   = {avail:>5.2f} GB  留给权重\n")
print(f"卸载比例 r = 1 - ({avail:.2f} - {P_DENSE_GB}) / {P_EXP_GB} = {r_real:.4f}  → {r_real * 100:.2f}%")
print(f"常驻专家 {resident_real:.2f} GB，每 token 仍需搬运 {d_real:.2f} GB "
      f"（比名义账本多 {(d_real - d24) / d24 * 100:.1f}%）")

assert r_real > r24                                    # 扣开销后卸载更多
assert avail > P_DENSE_GB                              # 至少装得下 dense 部分

## 3️⃣ 机制图

下图把「双缓冲槽」「两个硬件单元」「时序重叠」画在一起：**上层是搬运队列，下层是计算队列，计算整体向后错开一层，交错出来的部分就是 overlap，暴露出来的空白就是 stall。**

![](images/12_offload_pipeline.svg)

读图要点：

1. **搬运行首尾相连**（PCIe 链路一直在忙），**计算行整体右移一层** —— 这个错位就是流水；
2. 计算行里的橙色小块是 `stall`：搬运没按时到位，SM 空转；
3. 重叠区（紫色带）本质是「**搬运时间被计算掩盖**」，掩盖的前提是 `transfer ≤ compute`；一旦搬运比计算慢（图中 60 ms vs 45 ms），每层必然漏出一段 stall；
4. 深度 2 是**最小可行深度**：深度 1 没有第二个槽可用 → 零重叠；深度 3+ 收益递减但多吃一份显存（24 GB 卡上很贵）。

In [ ]:
def find_svg(name):
    '''兼容「在本目录运行」和「在仓库根运行」两种启动方式。'''
    for cand in (
        os.path.join("images", name),
        os.path.join("15-AI-Infra", "教学", "images", name),
        os.path.join("..", "images", name),
    ):
        if os.path.exists(cand):
            return cand
    return None


import xml.etree.ElementTree as ET

svg_path = find_svg("12_offload_pipeline.svg")
assert svg_path is not None, "机制图缺失：images/12_offload_pipeline.svg"
root = ET.parse(svg_path).getroot()
labels = " ".join(t.text or "" for t in root.iter() if t.tag.endswith("text"))

print("机制图路径 :", svg_path)
print("画布规格   :", root.get("width"), "×", root.get("height"), "| viewBox =", root.get("viewBox"))
print("文本标注数 :", sum(1 for t in root.iter() if t.tag.endswith("text")))
for kw in ("compute", "transfer", "stall", "overlap", "双缓冲"):
    assert kw in labels, f"机制图缺少关键标注：{kw}"
print("关键标注   : compute / transfer / stall / overlap / 双缓冲 —— 齐全")

# 用 ASCII 把同一段时间线再画一遍（列宽有限，用来核对「重叠区」的相对位置）
def ascii_timeline(n_layer=3, x_ms=60, c_ms=45, cols=66):
    span = n_layer * x_ms + c_ms
    scale = cols / span

    def put(row, t0, t1, ch):
        for k in range(int(round(t0 * scale)), int(round(t1 * scale))):
            if 0 <= k < cols:
                row[k] = ch

    tf, cp = [" "] * cols, [" "] * cols
    for l in range(n_layer):
        put(tf, l * x_ms, (l + 1) * x_ms, "#")               # 链路串行，首尾相连
        put(cp, (l + 1) * x_ms, (l + 1) * x_ms + c_ms, "=")  # 计算整体右移一层
    print("\n       " + "".join(str(i // 10 % 10) for i in range(cols)))
    print("搬运 # |" + "".join(tf) + "|")
    print("计算 = |" + "".join(cp) + "|")
    print("       每个 # 是搬运、每个 = 是计算；同一列两者都有 = 重叠，只有 # = GPU 在等(stall)")


ascii_timeline()

## 4️⃣ 手写实现：CPU-GPU 双缓冲流水模拟器

真实的流水在 GPU 上是「一个 kernel 在 SM 上跑 + 一堆 `cudaMemcpyAsync` 在 DMA 引擎上跑」，我们在 Python 里用**离散事件模拟**忠实复现它的时间关系：不需要线程，只需要正确地维护「队列串行 + 缓冲依赖」这两个约束。

### 建模约定

| 要素 | 含义 | 约束 |
|---|---|---|
| 搬运队列 | 每层专家从 CPU 到 GPU 的拷贝 | **串行**：一条 PCIe 链路，同一时刻只有一个拷贝 |
| 计算队列 | 每层在 SM 上的 forward | **串行**：同一时刻只有一层在算 |
| 缓冲槽 | 存放「已搬到的下一层专家」 | 深度 1 = 单缓冲；深度 2 = 双缓冲 |
| 依赖 | 计算第 $i$ 层前，第 $i$ 层专家必须已在显存 | $\text{compute\_start}_i \ge \text{transfer\_end}_i$ |
| 依赖 | 搬运第 $i$ 层前，槽已空 | 深度 2 时 $\text{transfer}_i$ 可与 $\text{compute}_{i-1}$ 并行 |

### 三种调度（本节要对比的对象）

| 调度 | 何时知道「下一层要哪些专家」 | 重叠来源 | 预期 |
|---|---|---|---|
| ① 逐层同步 `sync` | 算完当前层才知道 | 无（深度 1：搬完才算） | stall = 全部搬运时间 |
| ② 静态 layer-wise 预取 `static` | 提前一层（上一层路由已知 / 预取启发式） | 第 $i+1$ 层搬运与第 $i$ 层计算并行 | stall ≈ $\max(0, x - c)$ |
| ③ 动态路由触发 `dynamic` | 上一层算完才触发，且预测可能错 | 只有被预测中的那部分能提前搬 | 介于 ① ② 之间 |

### 递推式（深度 2，静态预取）

设第 $i$ 层搬运耗时 $x_i$、计算耗时 $c_i$，记 $[xs_i, xe_i]$ 为搬运区间、$[cs_i, ce_i]$ 为计算区间：

$$ xs_0 = 0,\quad xe_0 = x_0,\quad cs_0 = xe_0,\quad ce_0 = cs_0 + c_0 $$

$$ xs_i = \max(cs_{i-1},\, xe_{i-1}),\quad xe_i = xs_i + x_i,\quad cs_i = \max(ce_{i-1},\, xe_i) $$

第一个 $\max$ 体现两条队列的串行约束（搬运不能重叠），第二个 $\max$ 体现「等数据」的依赖 —— **stall 就是 $cs_i - ce_{i-1}$ 的正部**。

动态路由把 $x_i$ 拆成命中预测的 $h x_i$（可以从 $cs_{i-1}$ 提前搬）和漏掉的 $(1-h)x_i$（必须等 $ce_{i-1}$ 之后才发起）。

In [ ]:
class PipelineSimulator:
    '''CPU→GPU 专家搬运 与 GPU 计算 的双队列事件流模拟器（深度 1/2）。

    参数
    ----
    layer_bytes    : (L,) 每层需搬运的字节数（GB，已按驻留比例折算）
    layer_compute  : (L,) 每层 GPU 计算时间（s，已乘 batch）
    bw_gb_s        : PCIe 有效带宽 GB/s
    mode           : 'sync' | 'static' | 'dynamic'
    depth          : 缓冲深度；'sync' 强制按深度 1 处理
    hit_rate       : 仅 'dynamic' 使用，预测路由命中率（0~1）
    '''

    def __init__(self, layer_bytes, layer_compute, bw_gb_s,
                 mode="static", depth=2, hit_rate=0.85):
        assert mode in ("sync", "static", "dynamic")
        self.x = np.asarray(layer_bytes, dtype=float) / float(bw_gb_s)
        self.c = np.asarray(layer_compute, dtype=float)
        assert self.x.shape == self.c.shape and self.x.ndim == 1
        assert np.all(self.x >= 0) and np.all(self.c >= 0)
        self.mode = mode
        self.depth = 1 if mode == "sync" else min(int(depth), 2)
        self.hit_rate = float(hit_rate)
        self.L = len(self.x)
        self.trace = None          # (L, 4) = [搬运起, 搬运止, 计算起, 计算止]

    # ---------------- 排程 ----------------
    def schedule(self):
        L, x, c = self.L, self.x, self.c
        xs, xe = np.zeros(L), np.zeros(L)
        cs, ce = np.zeros(L), np.zeros(L)

        if self.depth == 1:
            # 单缓冲：槽只有一个 —— 上一层算完、槽才空，下一层才能开始搬 → 零重叠
            for i in range(L):
                xs[i] = 0.0 if i == 0 else ce[i - 1]
                xe[i] = xs[i] + x[i]
                cs[i] = xe[i]
                ce[i] = cs[i] + c[i]
        else:
            # 双缓冲：第 i 层搬运与第 i-1 层计算并行
            xs[0], xe[0] = 0.0, x[0]
            cs[0], ce[0] = xe[0], xe[0] + c[0]
            for i in range(1, L):
                if self.mode == "dynamic":
                    pred = self.hit_rate * x[i]      # 预测命中：可提前搬
                    miss = x[i] - pred               # 预测漏搬：只能事后搬
                    xs_pre = max(cs[i - 1], xe[i - 1])
                    xe_pre = xs_pre + pred
                    xs_miss = max(ce[i - 1], xe_pre)
                    xs[i], xe[i] = xs_pre, xs_miss + miss
                else:
                    xs[i] = max(cs[i - 1], xe[i - 1])
                    xe[i] = xs[i] + x[i]
                cs[i] = max(ce[i - 1], xe[i])
                ce[i] = cs[i] + c[i]

        self.trace = np.stack([xs, xe, cs, ce], axis=1)
        return self.trace

    # ---------------- 指标 ----------------
    def metrics(self, batch=1):
        if self.trace is None:
            self.schedule()
        xs, xe, cs, ce = self.trace.T
        prev_end = np.concatenate([[0.0], ce[:-1]])
        stall = float(np.sum(np.maximum(0.0, cs - prev_end)))   # GPU 等数据的空转时间

        makespan = float(ce[-1])
        transfer_busy = float(np.sum(self.x))
        compute_busy = float(np.sum(self.c))
        overlap = 1.0 - stall / transfer_busy if transfer_busy > 0 else 1.0

        assert abs(makespan - (compute_busy + stall)) < 1e-9 * max(1.0, makespan)  # 时间守恒
        return {
            "mode": self.mode,
            "depth": self.depth,
            "makespan_s": makespan,
            "compute_busy_s": compute_busy,
            "transfer_busy_s": transfer_busy,
            "stall_s": stall,
            "overlap_ratio": float(np.clip(overlap, 0.0, 1.0)),
            "avg_stall_ms_per_layer": stall / self.L * 1e3,
            "tokens_per_s": batch / makespan,
        }


def build_case(bw_gb_s, r, batch=1, mode="static", depth=2, hit_rate=0.85):
    '''按账本生成一个算例：每层搬运字节 = 每 token 搬运量 / 层数。'''
    layer_bytes = np.full(L_LAYERS, per_token_transfer_gb(r) / L_LAYERS)
    layer_compute = np.full(L_LAYERS, batch * (ACTIVE_GB / L_LAYERS) / HBM_BW_GB_S)
    return PipelineSimulator(layer_bytes, layer_compute, bw_gb_s,
                             mode=mode, depth=depth, hit_rate=hit_rate)


print("PipelineSimulator 定义完成；build_case 已封装账本 → 每层字节/计算时间。")

In [ ]:
REF_BW, REF_F, REF_BATCH = 32, 0.5, 8      # 参考工况：PCIe 32 GB/s、驻留一半、batch 8
REF_R = 1.0 - REF_F

schemes = [
    ("sync",    "①逐层同步（无重叠）",              1, 1.00),
    ("static",  "②静态 layer-wise 预取（深度2）",   2, 1.00),
    ("dynamic", "③动态路由触发（预测命中 0.85）",   2, 0.85),
]

print(f"参考工况：BW={REF_BW} GB/s，f={REF_F}（r={REF_R:.2f}），batch={REF_BATCH}，"
      f"每层搬运 {per_token_transfer_gb(REF_R) / L_LAYERS * 1e3:.1f} MB")
print(f"每层计算 {(REF_BATCH * ACTIVE_GB / L_LAYERS / HBM_BW_GB_S) * 1e3:.2f} ms\n")
print(f"{'调度':<30}{'makespan(s)':>13}{'stall(s)':>10}{'重叠率':>8}{'stall/层(ms)':>14}{'tokens/s':>10}")

demo = {}
for mode, label, depth, hit in schemes:
    sim = build_case(REF_BW, REF_R, REF_BATCH, mode=mode, depth=depth, hit_rate=hit)
    m = sim.metrics(REF_BATCH)
    demo[mode] = (sim, m)
    print(f"{label:<30}{m['makespan_s']:>13.4f}{m['stall_s']:>10.4f}"
          f"{m['overlap_ratio']:>8.3f}{m['avg_stall_ms_per_layer']:>14.2f}{m['tokens_per_s']:>10.2f}")

_, m_sync = demo["sync"]
assert abs(m_sync["overlap_ratio"]) < 1e-9                 # 逐层同步：零重叠
assert abs(m_sync["stall_s"] - m_sync["transfer_busy_s"]) < 1e-9 * m_sync["transfer_busy_s"]
print("\n①同步校验：重叠率 = 0，且 stall 恰等于总传输时间 PASS")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9.4, 4.8), sharex=True)
for ax, (mode, label) in zip(axes, [("sync", "①逐层同步：零重叠 —— 搬一段算一段交替（橙 = 等搬运）"),
                                    ("static", "②静态预取：双缓冲把搬运藏进计算"),
                                    ("dynamic", "③动态路由：命中预测的部分能藏，漏搬的仍暴露")]):
    sim, m = demo[mode]
    tr = sim.trace
    prev_end = 0.0
    for i in range(L_LAYERS):
        xs, xe, cs, ce = tr[i]
        ax.barh(2, xe - xs, left=xs, height=0.60, color="#93c5fd",
                edgecolor="#2563eb", linewidth=0.5)
        ax.barh(1, ce - cs, left=cs, height=0.60, color="#86efac",
                edgecolor="#16a34a", linewidth=0.5)
        if cs > prev_end + 1e-12:      # 计算开始前的空转 = stall
            ax.barh(1, cs - prev_end, left=prev_end, height=0.60, color="#fdba74",
                    edgecolor="#ea580c", linewidth=0.5)
        prev_end = ce
    ax.set_yticks([1, 2], ["GPU 计算", "PCIe 搬运"], fontsize=8)
    ax.set_ylabel("")
    ax.set_title(f"{label}   重叠率 {m['overlap_ratio']:.1%} · stall {m['stall_s']:.3f}s · "
                 f"{m['tokens_per_s']:.1f} tokens/s", fontsize=9, loc="left")
    ax.grid(axis="x", alpha=0.25)
    ax.set_xlim(0, demo["sync"][1]["makespan_s"] * 1.02)

axes[-1].set_xlabel("时间 (s)")
fig.suptitle(f"三种调度的 32 层流水甘特图（BW={REF_BW} GB/s, f={REF_F}, batch={REF_BATCH}）",
             fontsize=10)

from matplotlib.patches import Patch

fig.legend(
    handles=[
        Patch(facecolor="#93c5fd", edgecolor="#2563eb", label="PCIe 搬运（transfer）"),
        Patch(facecolor="#86efac", edgecolor="#16a34a", label="GPU 计算（compute）"),
        Patch(facecolor="#fdba74", edgecolor="#ea580c", label="stall：GPU 等数据空转"),
    ],
    loc="lower center", ncol=3, fontsize=8.5, frameon=False,
)
plt.tight_layout(rect=(0, 0.05, 1, 0.96))
plt.show()

## 5️⃣ 数值实验：重叠率 / stall / tokens-s

### 实验设置

| 维度 | 取值 | 目的 |
|---|---|---|
| PCIe 带宽 $BW$ | 8 / 16 / 32 / 64 GB/s | 覆盖 PCIe3 x16 ~ NVLink/HBM 级 |
| 驻留比例 $f$ | 0.3 / 0.5 / 0.7 | 覆盖「重度卸载」到「轻度卸载」 |
| 调度 | ① 同步 / ② 静态预取 / ③ 动态路由 | 重叠率的来源差异 |
| batch | 8（吞吐扫描）/ 1（stall 定律验证） | 分离「搬运瓶颈」与「算力瓶颈」 |

### 四条待验证断言

1. 重叠率(②) > 重叠率(①)：预取确实能吃掉搬运时间；
2. **$BW$ 翻倍时 stall 近似减半（±10%）**：只在「计算 ≪ 搬运」的搬运瓶颈区成立（下面会用条件筛出这些配对）；
3. stall(①) ≈ 总传输时间：零重叠调度的定义性质；
4. tokens/s 随 $BW$ 单调不减：带宽越多越快。

> 断言 2 的门槛要讲清楚：$stall \approx x - c$，所以 $BW$ 翻倍得到的是 $\frac{x/2 - c}{x - c}$ —— 只有 $c/x \le 5\%$ 时才近似 $1/2$。不设条件就断言「减半」是把结论用错了地方。

In [ ]:
BATCH_STEP = 8
tps_mat = np.zeros((len(BW_LIST), len(RES_LIST)))
stall_mat = np.zeros_like(tps_mat)
ovl_mat = np.zeros_like(tps_mat)
scan = {}

for i, bw in enumerate(BW_LIST):
    for j, f in enumerate(RES_LIST):
        sim = build_case(bw, 1.0 - f, BATCH_STEP, mode="static", depth=2)
        m = sim.metrics(BATCH_STEP)
        scan[(bw, f)] = (sim, m)
        tps_mat[i, j] = m["tokens_per_s"]
        stall_mat[i, j] = m["stall_s"]
        ovl_mat[i, j] = m["overlap_ratio"]

print(f"扫描完成：{len(BW_LIST)}×{len(RES_LIST)} 组（batch={BATCH_STEP}，静态预取）\n")
print(f"{'BW(GB/s)':>9} " + " ".join(f"{'f=' + str(f):>16}" for f in RES_LIST))
for i, bw in enumerate(BW_LIST):
    cells_s = " ".join(f"{tps_mat[i, j]:>7.2f}t/s {ovl_mat[i, j]:>6.1%}" for j in range(len(RES_LIST)))
    print(f"{bw:>9} {cells_s}")
print("\n（每格：tokens/s 与重叠率）")
print(f"算力天花板 {cpu_ceiling:.1f} tokens/s —— 高 BW × 高驻留的组合会撞上它。")

In [ ]:
fig, ax = plt.subplots(figsize=(6.8, 4.0))
im = ax.imshow(tps_mat, cmap="viridis", aspect="auto")
ax.set_xticks(range(len(RES_LIST)), [f"{f:.1f}" for f in RES_LIST])
ax.set_yticks(range(len(BW_LIST)), [f"{b}" for b in BW_LIST])
ax.set_xlabel("专家权重驻留比例 f = 1 - r")
ax.set_ylabel("PCIe 有效带宽 BW (GB/s)")
ax.set_title(f"静态预取下的等效吞吐（tokens/s，batch={BATCH_STEP}）", fontsize=10)

vmax = float(tps_mat.max())
for i in range(len(BW_LIST)):
    for j in range(len(RES_LIST)):
        ax.text(j, i, f"{tps_mat[i, j]:.1f}", ha="center", va="center", fontsize=9,
                color="white" if tps_mat[i, j] < vmax * 0.62 else "black")
fig.colorbar(im, ax=ax, label="tokens/s")
plt.tight_layout()

print("读图：纵向往上（带宽翻倍）与横向往右（多驻留）都能提吞吐，二者可互换；")
print(f"      右上角被 {cpu_ceiling:.1f} tokens/s 的算力天花板压平 —— 此时卸载已不再是瓶颈。")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.8, 3.8))

# 左：stall 随 BW 的变化（batch=1，纯粹看搬运瓶颈区的 1/BW 定律）
for f in RES_LIST:
    stalls = [build_case(bw, 1.0 - f, 1, mode="static").metrics(1)["stall_s"] * 1e3 for bw in BW_LIST]
    axes[0].plot(BW_LIST, stalls, marker="o", markersize=5, label=f"f={f}（静态预取）")
sync_stalls = [build_case(bw, 1.0 - 0.5, 1, mode="sync").metrics(1)["stall_s"] * 1e3 for bw in BW_LIST]
axes[0].plot(BW_LIST, sync_stalls, marker="s", linestyle="--", color="#ea580c", label="f=0.5（逐层同步）")
axes[0].set_xscale("log")
axes[0].set_xticks(BW_LIST, [str(b) for b in BW_LIST])
axes[0].minorticks_off()
axes[0].set_xlabel("PCIe 带宽 BW (GB/s)")
axes[0].set_ylabel("总 stall (ms)")
axes[0].set_title("stall 与带宽近似成反比（batch=1）", fontsize=10)
axes[0].grid(alpha=0.3)
axes[0].legend(fontsize=8)

# 右：重叠率随 BW 的变化（batch=8，能看到「搬运瓶颈 → 算力瓶颈」的过渡）
for mode, label, ls in [("sync", "①逐层同步", "--"), ("dynamic", "③动态路由", "-."), ("static", "②静态预取", "-")]:
    ovl = [build_case(bw, REF_R, BATCH_STEP, mode=mode, hit_rate=0.85).metrics(BATCH_STEP)["overlap_ratio"]
           for bw in BW_LIST]
    axes[1].plot(BW_LIST, [v * 100 for v in ovl], marker="o", markersize=5, linestyle=ls, label=label)
axes[1].set_xscale("log")
axes[1].set_xticks(BW_LIST, [str(b) for b in BW_LIST])
axes[1].minorticks_off()
axes[1].set_xlabel("PCIe 带宽 BW (GB/s)")
axes[1].set_ylabel("重叠率 (%)")
axes[1].set_ylim(-3, 105)
axes[1].set_title(f"重叠率：预取把小 stall 抹平（f={REF_F}, batch={BATCH_STEP}）", fontsize=10)
axes[1].grid(alpha=0.3)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
checks = []


def check(name, cond, detail=""):
    checks.append((name, bool(cond), detail))
    print(f"{'PASS' if cond else 'FAIL'} | {name} | {detail}")


# ---------- 断言 ①：静态预取的重叠率高于逐层同步 ----------
sim_s = build_case(REF_BW, REF_R, REF_BATCH, mode="sync")
sim_t = build_case(REF_BW, REF_R, REF_BATCH, mode="static")
sim_d = build_case(REF_BW, REF_R, REF_BATCH, mode="dynamic", hit_rate=0.85)
m1, m2, m3 = sim_s.metrics(REF_BATCH), sim_t.metrics(REF_BATCH), sim_d.metrics(REF_BATCH)

check("① 重叠率(②静态预取) > 重叠率(①逐层同步)",
      m2["overlap_ratio"] > m1["overlap_ratio"] + 0.05,
      f"{m1['overlap_ratio']:.3f} → {m2['overlap_ratio']:.3f}")
check("①b ①的重叠率为 0（零重叠定义性质）",
      0.0 <= m1["overlap_ratio"] <= 1e-9,
      f"{m1['overlap_ratio']:.3e}")

# ---------- 断言 ②：BW 翻倍 → stall 近似减半（仅在 compute/transfer ≤ 5% 的搬运瓶颈区）----------
pairs = []
for bw_lo in BW_LIST:
    bw_hi = bw_lo * 2
    if bw_hi not in BW_LIST:
        continue
    lo = build_case(bw_lo, 0.5, 1, mode="static").metrics(1)
    hi = build_case(bw_hi, 0.5, 1, mode="static").metrics(1)
    ratio_cx = lo["compute_busy_s"] / lo["transfer_busy_s"]
    if ratio_cx <= 0.05:                       # 搬运瓶颈区才适用 1/BW 定律
        pairs.append((bw_lo, bw_hi, hi["stall_s"] / lo["stall_s"], ratio_cx))

print(f"\n适用「减半定律」的配对（compute/transfer ≤ 5%）："
      f"{[(a, b) for a, b, _, _ in pairs]}")
for a, b, ratio, cx in pairs:
    print(f"   {a}→{b} GB/s：stall 比值 {ratio:.4f}（c/x={cx:.3f}）")
check("② BW 翻倍 → stall 近似减半（±10%）",
      len(pairs) >= 1 and all(0.45 <= ratio <= 0.55 for _, _, ratio, _ in pairs),
      f"{[(a, b, round(r, 3)) for a, b, r, _ in pairs]}")

# ---------- 断言 ③：①的 stall ≈ 总传输时间 ----------
check("③ stall(①逐层同步) ≈ 总传输时间",
      abs(m1["stall_s"] - m1["transfer_busy_s"]) <= 1e-9 * m1["transfer_busy_s"],
      f"stall={m1['stall_s']:.6f}s, transfer={m1['transfer_busy_s']:.6f}s")

# ---------- 断言 ④：tokens/s 随 BW 单调不减 ----------
mono = True
detail = []
for j, f in enumerate(RES_LIST):
    col = tps_mat[:, j]
    if not np.all(np.diff(col) >= -1e-9):
        mono = False
    detail.append(f"f={f}: " + " ≤ ".join(f"{v:.2f}" for v in col))
check("④ tokens/s 随 BW 单调不减", mono, "; ".join(detail))

# ---------- 附加断言：结构性质 ----------
check("附加 动态路由的 stall ≥ 静态预取（预测漏搬有代价）",
      m3["stall_s"] >= m2["stall_s"] - 1e-12,
      f"{m2['stall_s']:.4f} ≤ {m3['stall_s']:.4f}")
check("附加 重叠率落在 [0,1] 区间",
      all(0.0 <= v <= 1.0 for v in ovl_mat.ravel()),
      f"min={ovl_mat.min():.3f}, max={ovl_mat.max():.3f}")

# 吞吐不得超过解析上界（模拟器必须自洽）
worst = 0.0
for i, bw in enumerate(BW_LIST):
    for j, f in enumerate(RES_LIST):
        cap, _ = tps_bound(bw, 1.0 - f, BATCH_STEP)
        worst = max(worst, tps_mat[i, j] - cap)
check("附加 模拟吞吐 ≤ 解析上界 BW/(r·H)", worst <= 1e-9, f"最大超出 {worst:.3e} tokens/s")

n_fail = sum(1 for _, ok, _ in checks if not ok)
print(f"\n{'✅ 全部断言 PASS' if n_fail == 0 else '❌ 存在失败断言'} （{len(checks) - n_fail}/{len(checks)}）")
assert n_fail == 0

In [ ]:
# 上界到底紧不紧：逐层同步 vs 静态预取，各自吃到解析上界的百分之多少
print(f"{'BW':>4}{'f':>6}{'同步t/s':>10}{'预取t/s':>10}{'上界t/s':>10}{'达成率(预取)':>16}")
rows_gap = []
for bw in BW_LIST:
    for f in RES_LIST:
        r = 1.0 - f
        sync_m = build_case(bw, r, BATCH_STEP, mode="sync").metrics(BATCH_STEP)
        static_m = scan[(bw, f)][1]
        cap, _ = tps_bound(bw, r, BATCH_STEP)
        ratio = static_m["tokens_per_s"] / cap
        rows_gap.append((bw, f, sync_m["tokens_per_s"], static_m["tokens_per_s"], cap, ratio))
        print(f"{bw:>4}{f:>6.1f}{sync_m['tokens_per_s']:>10.2f}{static_m['tokens_per_s']:>10.2f}"
              f"{cap:>10.2f}{ratio:>15.1%}")

worst_sync = min(row[2] / row[4] for row in rows_gap)
best_static = max(row[5] for row in rows_gap)
print(f"\n逐层同步最低只达成上界的 {worst_sync:.1%} —— 这就是「不重叠」的代价；")
print(f"静态预取最高达成 {best_static:.1%} —— 双缓冲把上界基本吃满（上界是紧的）。")
assert best_static >= 0.95          # 静态预取应接近上界
assert worst_sync <= 0.9            # 逐层同步应明显低于上界

## 5️⃣（续）批量是把瓶颈推回算力的杠杆

前面 batch=1 时上界只有 1~12 tokens/s，看起来卸载等于判死刑。但注意上界式子里的不对称：

$$ \text{tokens/s} \le \min\left(\frac{B\cdot BW}{r\,H},\;\frac{\text{HBM}_{BW}}{A}\right) $$

- **搬运量不随 batch 增长**：同一批 token 往往命中相似的专家集合，一次搬运服务整个 batch；
- **计算量随 batch 线性增长**：$B$ 个 token 就多 $B$ 份访存与 FLOPs；
- 所以曲线先随 $B$ 线性上升（带宽瓶颈区），随后被 $\text{HBM}_{BW}/A$ 压平（算力瓶颈区）。

这就是卸载类系统的真实工程形态：**它牺牲的是单请求延迟，换的是聚合吞吐**。面试里被问「卸载能不能上生产」，正确回答是先问 batch/QPS 目标，再谈延迟预算。

In [ ]:
BATCHES = [1, 2, 4, 8, 16, 32]
tps_batch, bw_caps, cpu_caps = [], [], []

for bt in BATCHES:
    m = build_case(REF_BW, REF_R, bt, mode="static").metrics(bt)
    tps_batch.append(m["tokens_per_s"])
    cap, bw_cap = tps_bound(REF_BW, REF_R, bt)
    bw_caps.append(bw_cap)
    cpu_caps.append(cpu_ceiling)

bandwidth_limited = [b for b, v, c in zip(BATCHES, tps_batch, bw_caps) if v < cpu_ceiling - 1e-6]
print(f"{'batch':>6}{'tokens/s':>12}{'带宽上界':>11}{'算力上界':>11}{'瓶颈':>10}")
for b, v, c in zip(BATCHES, tps_batch, bw_caps):
    tag = "带宽" if c <= cpu_ceiling else "算力"
    print(f"{b:>6}{v:>12.2f}{c:>11.2f}{cpu_ceiling:>11.2f}{tag:>10}")

fig, ax = plt.subplots(figsize=(6.6, 3.8))
ax.plot(BATCHES, tps_batch, marker="o", color="#2563eb", label="静态预取实测吞吐")
ax.plot(BATCHES, bw_caps, linestyle="--", color="#93c5fd", label="带宽上界 B·BW/H")
ax.axhline(cpu_ceiling, linestyle=":", color="#ea580c", label=f"算力天花板 {cpu_ceiling:.1f}")
ax.set_xscale("log")
ax.set_xticks(BATCHES, [str(b) for b in BATCHES])
ax.set_xlabel("batch size")
ax.set_ylabel("tokens/s")
ax.set_title(f"批处理把瓶颈从 PCIe 推回算力（BW={REF_BW} GB/s, f={REF_F}）", fontsize=10)
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout()

ratio_gain = tps_batch[-1] / tps_batch[0]
print(f"\nbatch 1 → 32，吞吐提升 {ratio_gain:.1f}×（远小于 32×，因为中途撞上算力天花板）")
assert tps_batch[-1] > tps_batch[0] * 2            # 批量确实大幅提吞吐
assert all(v <= c + 1e-9 for v, c in zip(tps_batch, bw_caps))   # 不越上界
plt.show()

## 5️⃣（续）为什么「常驻热点专家」有效：路由本身的偏斜

账本里的 $f$ 假设「驻留的专家就是命中的专家」。现实中路由并不均匀：MoE 路由有明显的**热点专家 / 专家局部性**（同一个 token 位置、同一类输入反复命中少数专家）。所以：

- **驻留集合要按热度选**，而不是随机选同等数量 —— 同样的 $f$ 能覆盖更多命中；
- 这也解释了为什么 Fiddler 这类系统要有「**专家 popularity 统计 + 缓存淘汰策略**」；
- 极端情况：$f=0.5$ 时随机驻留只能覆盖 50% 命中，而按热度驻留可能覆盖 65%+ → 等效搬运量下降，吞吐直接受益。

下面用 Zipf 分布模拟不同偏斜程度下的**驻留命中率**（对路由分布做蒙特卡洛抽样）。

In [ ]:
def zipf_hit_rate(alpha, cache_frac, n_draw=60000, n_experts=N_EXPERTS):
    '''Zipf(alpha) 路由下，驻留 cache_frac 比例的最热专家能覆盖多少命中。'''
    w = 1.0 / np.arange(1, n_experts + 1) ** alpha
    w = w / w.sum()
    draws = np.random.choice(n_experts, size=n_draw, p=w)
    k = max(1, int(round(cache_frac * n_experts)))
    hot = np.argsort(-w)[:k]
    return float(np.isin(draws, hot).mean())


print(f"{'驻留比例 f':>10}" + "".join(f"{'α=' + str(a):>10}" for a in (0.0, 0.8, 1.2)))
for f in RES_LIST:
    line = "".join(f"{zipf_hit_rate(a, f):>10.1%}" for a in (0.0, 0.8, 1.2))
    print(f"{f:>10.1f}{line}")

fracs = np.linspace(0.125, 0.875, 7)
fig, ax = plt.subplots(figsize=(6.6, 3.6))
for alpha, color in zip((0.0, 0.8, 1.2), ("#94a3b8", "#2563eb", "#16a34a")):
    hits = [zipf_hit_rate(alpha, f) for f in fracs]
    ax.plot(fracs, [h * 100 for h in hits], marker="o", color=color,
            label=f"Zipf α={alpha}" + ("（均匀）" if alpha == 0 else ""))
ax.plot(fracs, fracs * 100, linestyle=":", color="#ea580c", label="无偏斜基线 y=x")
ax.set_xlabel("驻留专家比例 f")
ax.set_ylabel("驻留集合覆盖的命中率 (%)")
ax.set_title("路由偏斜越高，同样的显存覆盖越多命中", fontsize=10)
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout()

uniform = zipf_hit_rate(0.0, 0.5)
skewed = zipf_hit_rate(1.2, 0.5)
transfer_drop = 1.0 - (1.0 - skewed) / (1.0 - uniform)
print(f"\nf=0.5：均匀路由覆盖 {uniform:.1%}，强偏斜覆盖 {skewed:.1%}"
      f" → 命中率提升 {(skewed - uniform) / uniform * 100:.1f}%，同等吞吐下等效搬运量下降 {transfer_drop * 100:.1f}%")
assert 0.45 <= uniform <= 0.55                    # 均匀路由下应≈cache_frac
assert skewed > uniform                           # 偏斜提升命中率
plt.show()

## 6️⃣ 工程对照

| 系统 | 卸载对象 | 关键机制 | 与本节的对应 |
|---|---|---|---|
| **Fiddler**（ICLR'24） | 全部专家权重留 CPU，GPU 只放 KV/激活 | **CPU 侧调度 + GPU 侧计算流水**：用 CPU 上少量核做路由与拷贝调度，GPU 专注 GEMM；按 popularity 决定哪些专家常驻 | 对应 ②静态预取 + 热点驻留；它把「双缓冲队列」做进了系统层 |
| **DeepSpeed-MoE 推理** | 专家分片 / 张量并行 + 卸载 | 专家切片（EP）+ 层级调度；`MoE` 推理 kernel 融合 top-k 与 dispatch | 对应账本 $P_{\text{exp}}$ 的分摊：切到多卡后 $B$ 变大、$r$ 变小 |
| **KTransformers** | 稀疏 MoE 权重卸载到 CPU + 量化 | CPU 上跑专家（AMX/AVX512）+ GPU 跑 attention，**异构协同**而非纯搬运；用 `NUMA` 感知 + 分层缓存 | 把「搬运」变成「就地计算」，绕过 PCIe —— 上界式子里的 $BW$ 换成 CPU 内存带宽 |
| **量化卸载（QMoE / 2-bit）** | 专家权重压到 2~3 bit | 分组量化 + 非均匀码本，专家体积降 5~8× | 直接把 $P_{\text{exp}}$ 缩小 → $r$ 降低、每 token 搬运量同比下降 |
| **ZeRO-Offload**（训练态） | **优化器状态 + 梯度 + 参数** | 训练时把 Adam 状态与梯度放 CPU，按需换入换出 | 对照项：它是**训练态**（梯度/优化器状态主导），本节是**推理态**（权重主导），调度语义完全不同 |

### 与 ZeRO-Offload 的区别（高频追问）

| 维度 | ZeRO-Offload | 专家卸载（本节） |
|---|---|---|
| 阶段 | 训练 | 推理 |
| 卸载内容 | 优化器状态（最大头）、梯度、部分参数 | 专家权重 |
| 触发方式 | 按优化器 step 的**周期性**换入换出 | 按 **token 路由**的动态命中 |
| 是否可预测 | 完全可预测（固定顺序） | 部分可预测（路由热点） |
| 主要代价 | PCIe 带宽 + 计算/通信重叠难度 | 单 token 延迟（stall） |
| 优化重点 | 通信与计算重叠、bucket 化 | 预取、双缓冲、热点驻留、量化 |

一句话：**ZeRO-Offload 换的是「训练状态」，专家卸载换的是「推理权重」；前者靠 bucket 化重叠，后者靠路由预取重叠。**

### 常见坑（工程落地时）

1. **pageable memory 会毁掉重叠**：普通 CPU 内存的拷贝要走 bounce buffer 且可能隐式同步，必须用 `cudaHostAlloc`/pinned memory，否则 $BW$ 打五折还叠加额外同步；
2. **每层一次同步点就够了，别更多**：多余的 `streamSynchronize` 会把流水切碎；
3. **KV cache 会挤占预算**：长上下文场景下 $B$ 被 KV 吃掉，$r$ 反而上升（第 2️⃣ 节的预算拆分）；
4. **NUMA 与 CPU 侧拷贝线程**：拷贝线程绑核、避免跨 NUMA 访存；
5. **预测失败的代价不对称**：漏搬要多走一次完整搬运，误搬只是浪费带宽 —— 所以预取策略应偏保守（宁可多搬热点）。

In [ ]:
# 量化「重叠被内存类型吃掉多少」：pinned vs pageable + 额外同步开销
def effective_case(bw_gb_s, r, batch, eta, overhead_ms_per_layer):
    '''eta: 有效带宽折扣（pinned=1.0）；overhead_ms_per_layer: 每层额外同步开销。'''
    layer_bytes = np.full(L_LAYERS, per_token_transfer_gb(r) / L_LAYERS)
    layer_compute = np.full(
        L_LAYERS,
        batch * (ACTIVE_GB / L_LAYERS) / HBM_BW_GB_S + overhead_ms_per_layer / 1e3,
    )
    return PipelineSimulator(layer_bytes, layer_compute, bw_gb_s * eta,
                             mode="static", depth=2)


cases = [
    ("pinned memory + 双 stream", 1.00, 0.00),
    ("pinned + 每层一次 Sync", 1.00, 0.80),
    ("pageable memory（隐式同步）", 0.55, 0.80),
]
base_tps = None
print(f"{'内存/同步方式':<28}{'stall(s)':>10}{'重叠率':>8}{'tokens/s':>10}{'相对':>8}")
for label, eta, ovh in cases:
    m = effective_case(REF_BW, REF_R, REF_BATCH, eta, ovh).metrics(REF_BATCH)
    if base_tps is None:
        base_tps = m["tokens_per_s"]
    print(f"{label:<28}{m['stall_s']:>10.4f}{m['overlap_ratio']:>8.3f}"
          f"{m['tokens_per_s']:>10.2f}{m['tokens_per_s'] / base_tps:>8.1%}")

pin_only = effective_case(REF_BW, REF_R, REF_BATCH, 1.00, 0.0).metrics(REF_BATCH)
pageable = effective_case(REF_BW, REF_R, REF_BATCH, 0.55, 0.8).metrics(REF_BATCH)
print(f"\n结论：仅换成 pageable memory，吞吐掉到 {pageable['tokens_per_s'] / pin_only['tokens_per_s']:.0%}"
      f" —— 这是卸载系统最常见的「代码没问题但就是慢」的根因。")
assert pageable["tokens_per_s"] < pin_only["tokens_per_s"]
assert pin_only["overlap_ratio"] <= 1.0

### 三条技术路线的一句话选型

| 路线 | 代表 | 一句话 | 适用场景 |
|---|---|---|---|
| **原地算**（异构协同） | KTransformers | 专家留在 CPU 上用 AVX512/AMX 就地算，绕开 PCIe | CPU 强、batch 小、延迟敏感 |
| **搬过来算**（流水卸载） | Fiddler / DeepSpeed-MoE | GPU 专注计算，CPU 侧专门做调度与拷贝 | GPU 强、追求聚合吞吐 |
| **压小了再搬**（量化卸载） | QMoE / 2-bit 专家 | 体积降 5~8×，带宽需求同比下降 | 显存与带宽双紧、能接受精度损失 |

生产形态通常是**组合拳**：量化压缩 $P_{\text{exp}}$ + 热点专家常驻 + 双缓冲预取 + pinned memory —— 四个杠杆分别作用在账本的 $P_{\text{exp}}$、$r$、$stall$ 和有效 $BW$ 上。

## 🎯 面试考点

1. **为什么要卸载**：MoE 参数集中在专家，推理只激活 top-k；显存装不下全量专家，卸载是用带宽换显存；
2. **卸载比例怎么定**：$r = 1-(B-P_{\text{dense}})/P_{\text{exp}}$，并且 $B$ 要扣掉 KV cache / 激活 / 上下文开销；
3. **吞吐上界**：$\text{tokens/s}\le BW/(rH)$，与算力上限 $\text{HBM}_{BW}/A$ 取小；$r$ 与 $BW$ 等价可换；
4. **重叠怎么来**：双缓冲 + 双层流水，搬运走 DMA、计算走 SM；深度 2 是最小可行深度；
5. **三种调度差异**：逐层同步（零重叠，stall=全部搬运）→ 静态预取（stall≈$\max(0,x-c)$）→ 动态路由（预测漏搬有代价）；
6. **批量为什么救场**：搬运量不随 batch 增长、计算量线性增长，瓶颈从 PCIe 推回算力；
7. **热点专家**：路由偏斜 ⇒ 同样 $f$ 覆盖更多命中，必须按 popularity 选驻留集合并做淘汰；
8. **工程实现要点**：pinned memory、单层单同步点、NUMA 绑核、预测偏保守；
9. **量化**：2-bit 让 $P_{\text{exp}}$ 降 8×，等价于带宽提 8×（QMoE）；
10. **对照组**：ZeRO-Offload 是训练态（优化器状态/梯度），专家卸载是推理态（权重），调度语义不同。

### 追问集（能展开 30 秒就够）

- 深度 2 和深度 4 的差距在哪？为什么不是越深越好？（显存成本 + 收益递减 + 路由不确定性）
- 如果一层内多个 token 命中不同专家，搬运单元什么样？（按专家聚合 → 一次 copy 服务整个 batch）
- 预测器怎么做？（上一层路由外推、专家 popularity、轻量 router 预热）
- 卸载和 EP（专家并行）能不能同时用？（能；EP 降低单卡 $P_{\text{exp}}$，卸载降低剩余压力）
- PCIe 4.0 x16 vs NVLink 80 GB/s 对上界的影响？（$BW$ 线性放大，直到撞算力天花板）

## ✅ 自测清单 + 衔接

**手算能力**

- [ ] 给定 $B=24$、$P_{\text{dense}}=1.9$、$P_{\text{exp}}=45.1$，30 秒内算出 $r \approx 51\%$；
- [ ] 算出该配置每 token 搬运量（≈5.52 GB）与 @32 GB/s 的吞吐上界（≈5.8 tokens/s）；
- [ ] 解释为什么 $B$ 要扣掉 KV cache，并说明扣完后 $r$ 变大还是变小。

**代码能力**

- [ ] 不看代码写出深度 2 的递推式 $xs_i=\max(cs_{i-1}, xe_{i-1})$，并说明两项各约束什么；
- [ ] 说清 `stall = cs[i] - ce[i-1]` 为什么就是 GPU 空转时间；
- [ ] 复现断言②：解释「BW 翻倍 stall 减半」为什么需要 $c/x \le 5\%$ 的前提；
- [ ] 把 `hit_rate` 调到 1.0，验证动态路由退化成静态预取。

**表达与工程**

- [ ] 用「书架只放当天要读的书」讲清卸载，用「后厨备菜与炒菜」讲清重叠（各 60 秒）；
- [ ] 对比 Fiddler / DeepSpeed-MoE / KTransformers 三条技术路线各自绕开了什么；
- [ ] 说出 ZeRO-Offload 与专家卸载的 3 个本质区别；
- [ ] 列 3 个「代码没问题但就是慢」的坑（pageable memory、多余同步点、NUMA）。

**衔接**

- 上游：05-KV Cache 与推理内存（预算 $B$ 的构成）、07-量化（压缩 $P_{\text{exp}}$）、02-混合精度与显存账本（同一套账本方法）；
- 下游：06-推理引擎（continuous batching 决定 batch 上限）、04-集合通信（若用 EP/NVLink 替代 PCIe，$BW$ 换成互连带宽）；
- 同目录：10-FlashAttention（另一个「带宽受限」的经典算子，可对照 Roofline 视角）、03-分布式并行（ZeRO 家族）。